# Local Editing + Colab Execution Setup

**Use Case**: Edit code locally (VSCode/PyCharm), execute on Colab GPU

**Run this once** when connecting to a new Colab runtime

---

## Step 1: Clone/Pull Latest Code from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/Echo-Lee/RAG-embedding.git"
REPO_NAME = "RAG-embedding"

# Clone or pull latest
if os.path.exists(f'/content/{REPO_NAME}'):
    print(f"📥 Repository exists, pulling latest changes...")
    !cd /content/{REPO_NAME} && git pull
else:
    print(f"📦 Cloning repository...")
    !git clone {REPO_URL} /content/{REPO_NAME}

print(f"✅ Code ready at: /content/{REPO_NAME}")

## Step 2: Install Dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu gradio pyyaml openai tqdm

import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

## Step 4: Link Data from Drive

In [ ]:
import os

os.chdir(f'/content/{REPO_NAME}')

# Create data directories
!mkdir -p data/processed

# Link data from Drive
DRIVE_DATA_PATH = "/content/drive/MyDrive/Capstone-Data"

!ln -sf {DRIVE_DATA_PATH}/hospital data/processed/hospital
!ln -sf {DRIVE_DATA_PATH}/corruption data/processed/corruption

# Verify data
print("\n📊 Data files:")
!ls -lh data/processed/hospital/ 2>/dev/null || echo "⚠️  Hospital data not found in Drive"
!ls -lh data/processed/corruption/ 2>/dev/null || echo "⚠️  Corruption data not found in Drive"

## Step 5: Link Outputs to Drive (Auto-save)

In [ ]:
# Link outputs to Drive so indexes persist
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/Capstone-Outputs"
!mkdir -p {DRIVE_OUTPUT_PATH}

!rm -rf outputs
!ln -s {DRIVE_OUTPUT_PATH} outputs

print(f"✅ Outputs linked to: {DRIVE_OUTPUT_PATH}")
print("📝 All indexes will auto-save to Google Drive")

## Step 6: Configure API Keys

In [ ]:
from google.colab import userdata
import yaml

# Get API keys from Colab Secrets
# (Go to 🔑 Secrets in left sidebar to add AZURE_API_KEY and AZURE_ENDPOINT)

try:
    api_key = userdata.get('AZURE_API_KEY')
    endpoint = userdata.get('AZURE_ENDPOINT')
    print("✅ Retrieved API keys from Colab Secrets")
except:
    print("⚠️  Colab Secrets not found")
    api_key = input("Enter Azure API Key: ")
    endpoint = input("Enter Azure Endpoint: ")

# Create config files from templates
for dataset in ['hospital', 'corruption']:
    template_path = f'experiments/{dataset}_base_template.yaml'
    config_path = f'experiments/{dataset}_base.yaml'
    
    with open(template_path, 'r') as f:
        config = yaml.safe_load(f)
    
    config['azure_api_key'] = api_key
    config['azure_endpoint'] = endpoint
    
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"✅ Created {config_path}")

## Step 7: Add to Python Path

In [ ]:
import sys
from pathlib import Path

# Add src to Python path
PROJECT_ROOT = Path('/content') / REPO_NAME
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f"✅ Added to path: {PROJECT_ROOT / 'src'}")
print(f"\n📂 Current directory: {os.getcwd()}")

## ✅ Setup Complete!

**Now you can**:

1. **Import project modules** in any cell:
   ```python
   from config.config import load_config
   from data.loader import DataLoader
   from retrieval.retriever import HybridRetriever
   # etc.
   ```

2. **Edit code locally**, commit to GitHub, then:
   ```python
   !cd /content/RAG-embedding && git pull
   ```

3. **Run your pipeline**:
   ```python
   config = load_config('experiments/hospital_base.yaml')
   # ... your code
   ```

---

**Key Benefits**:
- ✅ Local editing with your favorite IDE
- ✅ Colab GPU for execution
- ✅ Data from Google Drive
- ✅ Outputs auto-save to Drive
- ✅ Code synced via GitHub